# Deepfake Detection using Vision Transformer (ViT)
### MCA Major Project — Fully Documented Notebook

This notebook walks through the complete pipeline: dataset loading, EDA,
preprocessing, augmentation, transfer-learning fine-tuning of a pretrained
ViT-B/16, evaluation, and single-image prediction.

**Note:** this notebook drives the same reusable modules used by
`train.py`, `test.py`, `predict.py` and `app.py` (`config.py`, `src/*.py`,
`utils.py`), so anything you change here should ideally be changed there too.

Run this notebook **from the project root** (`Deepfake-Detection-ViT/`),
after placing your dataset inside `dataset/train`, `dataset/validation`
and `dataset/test` (see `README.md`).


## 1. Introduction

Deepfakes are synthetically generated or manipulated media that can convincingly swap or alter a person's face and expressions. This project fine-tunes a **pretrained Vision Transformer (ViT-B/16)** using **transfer learning** to classify face images as **Real** or **Fake**, and wraps the trained model in an explainable, interactive Streamlit application.

## 2. Import Libraries

In [ ]:
import os
import sys

# Ensure the project root is importable when this notebook runs from notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path and os.path.basename(os.getcwd()) == "notebooks":
    sys.path.insert(0, PROJECT_ROOT)

import config
from utils import set_seed, get_logger, count_parameters
from src.dataset import get_datasets, get_dataloaders, dataset_summary
from src.eda import (
    print_dataset_overview, plot_class_distribution,
    plot_resolution_distribution, plot_random_samples, plot_pixel_distribution,
)
from src.model import build_model, get_param_groups
from src.metrics import (
    compute_metrics, save_classification_report, plot_confusion_matrix,
    plot_roc_curve, plot_pr_curve, plot_training_curves,
)
from src.gradcam import generate_gradcam, generate_attention_rollout, overlay_heatmap_on_image
from predict import predict_image

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from PIL import Image

set_seed()
logger = get_logger("notebook")
print(f"Device: {config.DEVICE} | Mixed precision available: {config.USE_AMP}")


## 3. Dataset

The dataset must follow the `ImageFolder` layout: `dataset/{train,validation,test}/{real,fake}/*.jpg`. `get_datasets()` validates that every split/class folder is populated before returning the datasets.

In [ ]:
print_dataset_overview()

datasets_dict = get_datasets()
summary = dataset_summary(datasets_dict)
summary


## 4. Exploratory Data Analysis (EDA)

Generates and saves: class distribution (bar + pie), image resolution scatter, random sample grid, and pixel-intensity histograms.

In [ ]:
plot_class_distribution()
plot_resolution_distribution()
plot_random_samples()
plot_pixel_distribution()

print(f"All EDA plots saved under: {config.EDA_REPORT_DIR}")


### 4.1 Visualize a saved EDA plot inline

In [ ]:
img_path = os.path.join(config.EDA_REPORT_DIR, "class_distribution.png")
if os.path.exists(img_path):
    plt.figure(figsize=(10, 4))
    plt.imshow(Image.open(img_path))
    plt.axis("off")
    plt.show()
else:
    print("Run the EDA cell above first.")


## 5. Preprocessing

All images are resized to `224x224`, converted to tensors, and normalized using the mean/std expected by the configured ViT backbone (see `src/dataset.py::_mean_std`).

In [ ]:
from src.dataset import build_eval_transforms
eval_transform = build_eval_transforms()
print(eval_transform)


## 6. Data Augmentation

Training-only augmentations: random horizontal flip, random rotation, random crop (with padding), color jitter, and random erasing (configured centrally in `config.AUGMENTATION`).

In [ ]:
from src.dataset import build_train_transforms
train_transform = build_train_transforms()
print(train_transform)


## 7. DataLoaders

In [ ]:
loaders = get_dataloaders()
print(f"Train batches: {len(loaders['train'])}")
print(f"Validation batches: {len(loaders['validation'])}")
print(f"Test batches: {len(loaders['test'])}")


## 8. Model — Pretrained ViT-B/16 with Transfer Learning

The backbone (`google/vit-base-patch16-224`) is loaded with ImageNet-pretrained weights and **frozen**; only a new classifier head (LayerNorm → Linear → GELU → Dropout → Linear) is trained initially. Call `model.unfreeze_last_n_blocks()` for a later fine-tuning stage.

In [ ]:
model = build_model()
param_counts = count_parameters(model)
print(f"Total parameters:     {param_counts['total']:,}")
print(f"Trainable parameters: {param_counts['trainable']:,}")
print(f"Frozen parameters:    {param_counts['frozen']:,}")


## 9. Training

Optimizer: **AdamW** with discriminative learning rates (higher LR for the new head, lower LR for any unfrozen backbone layers). Loss: **CrossEntropyLoss** with label smoothing. Scheduler: **CosineAnnealingLR**. **Mixed precision (AMP)** is used automatically when CUDA is available, with **early stopping** and **best-checkpoint saving** on validation loss.

For a full run, prefer the command line (better progress bars, resumability): `python train.py`. The cell below runs a short demo loop directly in the notebook.

In [ ]:
from utils import EarlyStopping, save_checkpoint, save_json

def run_epoch(model, loader, criterion, optimizer, scaler, train: bool):
    model.train(mode=train)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="train" if train else "val", leave=False):
        images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=config.USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            if train:
                if config.USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (torch.argmax(logits, dim=1) == labels).sum().item()
        total += images.size(0)
    return {"loss": total_loss / total, "acc": correct / total}


criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(get_param_groups(model), weight_decay=config.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.LR_SCHEDULER_T_MAX, eta_min=config.LR_SCHEDULER_ETA_MIN
)
scaler = torch.cuda.amp.GradScaler(enabled=config.USE_AMP)
early_stopping = EarlyStopping()

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")

DEMO_EPOCHS = 2  # increase for a real run, or use `python train.py`

for epoch in range(1, DEMO_EPOCHS + 1):
    train_metrics = run_epoch(model, loaders["train"], criterion, optimizer, scaler, train=True)
    val_metrics = run_epoch(model, loaders["validation"], criterion, optimizer, scaler, train=False)
    scheduler.step()

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["acc"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])

    print(f"Epoch {epoch}: train_loss={train_metrics['loss']:.4f} "
          f"train_acc={train_metrics['acc']:.4f} val_loss={val_metrics['loss']:.4f} "
          f"val_acc={val_metrics['acc']:.4f}")

    save_checkpoint(model, optimizer, epoch, val_metrics, config.LAST_MODEL_PATH)
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        save_checkpoint(model, optimizer, epoch, val_metrics, config.BEST_MODEL_PATH)

    if early_stopping.step(val_metrics["loss"]):
        print("Early stopping triggered.")
        break

save_json(history, config.TRAINING_HISTORY_PATH)


### 9.1 Training Curves

In [ ]:
if len(history["train_loss"]) > 0:
    plot_training_curves(history)
    plt.figure(figsize=(10, 4))
    plt.imshow(Image.open(config.TRAINING_CURVES_PATH))
    plt.axis("off")
    plt.show()


## 10. Evaluation

Computes accuracy, precision, recall, F1, ROC-AUC, confusion matrix, classification report, ROC curve and Precision-Recall curve on the held-out test split. For the full test-set evaluation, prefer `python test.py`.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    pos_idx = config.CLASS_TO_IDX["real"]
    for images, labels in tqdm(loader, desc="test", leave=False):
        images = images.to(config.DEVICE)
        probs = F.softmax(model(images), dim=1)
        preds = torch.argmax(probs, dim=1)
        y_true.extend(labels.tolist())
        y_pred.extend(preds.cpu().tolist())
        y_prob.extend(probs[:, pos_idx].cpu().tolist())
    return y_true, y_pred, y_prob


y_true, y_pred, y_prob = evaluate(model, loaders["test"])
metrics = compute_metrics(y_true, y_pred, y_prob)
metrics


In [ ]:
save_classification_report(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred)
plot_roc_curve(y_true, y_prob)
plot_pr_curve(y_true, y_prob)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, ["confusion_matrix.png", "roc_curve.png", "pr_curve.png"]):
    ax.imshow(Image.open(os.path.join(config.PLOTS_DIR, name)))
    ax.axis("off")
plt.tight_layout()
plt.show()


## 11. Prediction

Reuses `predict.py::predict_image()` so the notebook, CLI and Streamlit app all share identical inference logic.

In [ ]:
# Pick any test image to try prediction on
sample_class = config.CLASS_NAMES[0]
sample_dir = os.path.join(config.TEST_DIR, sample_class)
sample_files = [f for f in os.listdir(sample_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

if sample_files:
    sample_path = os.path.join(sample_dir, sample_files[0])
    image = Image.open(sample_path)
    result = predict_image(image, checkpoint_path=config.BEST_MODEL_PATH)

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted: {result['label'].upper()} ({result['confidence']*100:.1f}%)")
    plt.show()
    print(result)
else:
    print("No sample images found under", sample_dir)


## 12. Explainable AI — Grad-CAM & Attention Rollout

In [ ]:
if sample_files:
    from src.dataset import build_eval_transforms
    tensor = build_eval_transforms()(image.convert("RGB")).unsqueeze(0).to(config.DEVICE)

    heatmap, pred_class = generate_gradcam(model, tensor)
    overlay = overlay_heatmap_on_image(image, heatmap)

    plt.figure(figsize=(4, 4))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title("Grad-CAM Overlay")
    plt.show()

    if config.MODEL_BACKEND == "huggingface":
        with torch.no_grad():
            _, attentions = model(tensor, output_attentions=True)
        attn_heatmap = generate_attention_rollout(attentions)
        attn_overlay = overlay_heatmap_on_image(image, attn_heatmap)

        plt.figure(figsize=(4, 4))
        plt.imshow(attn_overlay)
        plt.axis("off")
        plt.title("Attention Rollout Overlay")
        plt.show()


## 13. Conclusion

This notebook demonstrated the full pipeline for fine-tuning a pretrained Vision Transformer for deepfake image classification: dataset loading and validation, exploratory data analysis, preprocessing and augmentation, transfer-learning fine-tuning with a frozen backbone and a lightweight classifier head, comprehensive evaluation, single-image prediction, and Grad-CAM/attention-based explainability.

**Next steps:** run the full training/evaluation via `python train.py` and `python test.py` on your complete dataset, then launch the interactive demo with `streamlit run app.py`. See `README.md` for the full project workflow, and `reports/` for the project report and presentation slides.